In [ ]:
!ls ../../data/

In [ ]:
# imports

%matplotlib inline

# import codecs
import pandas
import glob
from datetime import datetime
from collections import defaultdict
import numpy as np

# import networkx as nx
# import plotly.graph_objects as go
import plotly.express as px

In [ ]:
glob.glob('../../data/memes_infections*.log')

In [ ]:
PATIENT_0 = 'u/woodendoors7'

In [ ]:
# doc = codecs.open('../data/memes_infections.20.20.log','rU','UTF-16')
all_files = glob.glob('../../data/memes_infections*.log')
get_file = lambda fname: pandas.read_csv(
    fname,
    sep='\t',
    header=None,
    names=['post_dt',
            'infectee_name',
            'infectee_post_id',
            'infector_name',
            'infector_post_id',
            'comment_or_submission'],
    parse_dates=['post_dt']
)
df = pandas.concat([get_file(f) for f in all_files])
df.sort_values('post_dt', inplace=True)
print(df.size)
df.head()

In [ ]:
# [row for index, row in df.iterrows() if row.infectee_name == PATIENT_0]

In [ ]:
# list(df.iterrows())[0][1].to_dict()
# df[:2]

In [ ]:
# infection_graph = nx.DiGraph()

# for index, row in df.iterrows():
#     infection_graph.add_node(row.infector_name)
#     infection_graph.add_node(row.infectee_name)
#     infection_graph.add_edge(row.infector_name, row.infectee_name, **row.to_dict())
# infection_graph.edges(data=True)

In [ ]:
infectee_infector = {}
for index, row in df.iterrows():
    infectee_infector[row.infectee_name] = row.infector_name

In [ ]:
len(infectee_infector)

In [ ]:
data = {
    'infectee': [],
    'infector': [],
    'value': []
}
processed = set()

def get_patient_0(user):
    if user not in infectee_infector or user == PATIENT_0:
        return user
    else:
        return get_patient_0(infectee_infector[user])

valid_infectors = {}
valid_infectors[np.NaN] = []
invalid_infectors = set()
valid = set()
count = 0
for index, row in df.iterrows():
    count+=1
    processed = set()
    source_pat_0 = get_patient_0(row.infector_name)
    if source_pat_0 == PATIENT_0 or (row.infector_name is np.NaN):
        if row.infector_name in valid_infectors: # and len(valid_infectors[row.infector_name]) <= 20:
            valid_infectors[row.infector_name].append(row.infectee_name)
            valid_infectors[row.infectee_name] = []
            data['infector'].append(row.infector_name if row.infector_name is not np.NaN else '')
            data['infectee'].append(row.infectee_name)
            data['value'].append(1)
    else:
        invalid_infectors.add(source_pat_0)
#     if count == 3000:
#         print(row.to_dict())
#         break

print(len(invalid_infectors))
# from pprint import pprint; pprint(data)

# fig = px.sunburst(
#     data,
#     names='infectee',
#     parents='infector',
#     values='value',
# )
# fig.show()

In [ ]:
invalid_infectors

In [ ]:
import plotly.io as pio
# pio.orca.shutdown_server()
# pio.orca.config.executable = '/home/pmr3mh/miniconda3/lib/orca_app/orca'
# pio.orca.config.server_url = None
# pio.orca.config.port = None
# pio.orca.config.save()
# pio.orca.ensure_server()
# pio.orca.config
pio.orca.status

In [ ]:
# import subprocess
# subprocess.Popen(["/home/pmr3mh/miniconda3/lib/orca_app/orca", "-p", "--plotly", "--graph-only", "--mathjax", "serve", "37587", "/home/pmr3mh/.virtualenvs/nd230-ml-tensorflow-p1/lib/python3.5/site-packages/plotly/package_data/plotly.min.js", "https://cdnjs.cloudflare.com/ajax/libs/mathjax/2.7.5/MathJax.js", "--keep-alive"], stdout=subprocess.PIPE, shell=True)
timestamp = datetime.utcnow().isoformat()
print(datetime.utcnow().isoformat())
fig.write_image('plot_{}.png'.format(timestamp), width=800, height=800, scale=5)

In [ ]:
print(datetime.utcnow().isoformat())

In [ ]:
test_infector = 'u/5amukai'
processed = set()
hier = []
def get_patient_0(user):
    hier.append(user)
    if user in processed:
        print(user)
        return None
    else:
        processed.add(user)
    if user not in infectee_infector or user == PATIENT_0:
        return user
    else:
        return get_patient_0(infectee_infector[user])
print(get_patient_0(test_infector))
print(hier)
# print(len(valid_infectors[hier[1]]))
# print(test_infector in valid_infectors[hier[1]])
print(test_infector in valid_infectors)
print(test_infector in data['infector'])
print(test_infector in data['infectee'])
df.loc[(df['infector_name'] == test_infector) | (df['infectee_name'] == test_infector)]